In [7]:
import datetime

from openpyxl import Workbook
from openpyxl.styles import Alignment, Font

# ---- 請求先情報・商品情報（必要に応じて書き換えてください） ----
company_name = "株式会社ABC"
address = "〒101-0022 東京都千代田区神田練塀町300"
tel_fax = "TEL:03-1234-5678 FAX:03-1234-5678"
staff_name = "鈴木一郎"
invoice_no = "0001"

products = [
    {"name": "商品A", "quantity": 2, "unit_price": 10000},
    {"name": "商品B", "quantity": 1, "unit_price": 15000},
]

tax_rate = 0.1

# ---- ワークブック・シート作成 ----
wb = Workbook()
ws = wb.active
ws.title = "請求書"

# 列幅の設定
column_widths = {"A": 3, "B": 14, "C": 8, "D": 12, "E": 12, "F": 8, "G": 12}
for col, width in column_widths.items():
    ws.column_dimensions[col].width = width

# ---- タイトル ----
ws["B2"] = "請求書"
ws["B2"].font = Font(size=16, bold=True)

# ---- 発行元・宛先情報 ----
ws["B4"] = company_name
ws["B5"] = address
ws["B6"] = tel_fax
ws["B7"] = f"担当者名:{staff_name} 様"

ws["F4"] = "No."
ws["G4"] = invoice_no
ws["F5"] = "日付"
ws["G5"] = datetime.date.today().strftime("%Y/%m/%d")

# ---- 商品テーブルの見出し ----
headers = ["商品名", "数量", "単価", "金額"]
header_row = 10
for i, header in enumerate(headers):
    cell = ws.cell(row=header_row, column=2 + i, value=header)
    cell.font = Font(bold=True)
    cell.alignment = Alignment(horizontal="center")

# ---- 商品明細 ----
start_row = header_row + 1
for i, product in enumerate(products):
    row = start_row + i
    ws.cell(row=row, column=2, value=product["name"])
    ws.cell(row=row, column=3, value=product["quantity"])
    ws.cell(row=row, column=4, value=product["unit_price"])
    ws.cell(row=row, column=5, value=f"=C{row}*D{row}")

end_row = start_row + len(products) - 1
subtotal_row = end_row + 1
ws.cell(row=subtotal_row, column=5, value=f"=SUM(E{start_row}:E{end_row})")

# ---- 合計・消費税・税込合計 ----
total_row = subtotal_row + 2
tax_row = total_row + 1
grand_total_row = tax_row + 1

ws.cell(row=total_row, column=2, value="合計")
ws.cell(row=total_row, column=5, value=f"=E{subtotal_row}")

ws.cell(row=tax_row, column=2, value="消費税")
ws.cell(row=tax_row, column=5, value=f"=ROUND(E{total_row}*{tax_rate},0)")

ws.cell(row=grand_total_row, column=2, value="税込合計")
ws.cell(row=grand_total_row, column=5, value=f"=E{total_row}+E{tax_row}")

# 金額列を数値表示（3桁区切り）にする
for row in list(range(start_row, subtotal_row + 1)) + [total_row, tax_row, grand_total_row]:
    ws.cell(row=row, column=5).number_format = "#,##0"
for row in range(start_row, end_row + 1):
    ws.cell(row=row, column=4).number_format = "#,##0"

# ---- ファイル保存（ファイル名に現在日付を使用） ----
today_str = datetime.date.today().strftime("%Y%m%d")
filename = f"請求書_{today_str}.xlsx"
wb.save(filename)

print(f"{filename} を作成しました。")



請求書_20260906.xlsx を作成しました。
